# Laboratorium terbuka: difusi dan muka Fisher–KPP

Notebook ini merupakan pendamping komputasi mandiri untuk Bab 9. Seluruh perhitungan memakai NumPy, SciPy, dan Matplotlib, dapat dijalankan secara luring, serta tidak membaca berkas, data tersembunyi, maupun jaringan. Notebook ini bukan port atau rekonstruksi kode GUI MATLAB maupun PPLANE yang digunakan untuk membuat gambar sumber; kode aplikasi tersebut tidak tersedia dalam unit sumber beku.

Eksperimen pertama membangun gerak acak kisi dua dimensi dan menguji label yang telah dikoreksi, yaitu rata-rata kuadrat jarak, \(\mathbb{E}[L^2]\), bukan kuadrat dari jarak rata-rata. Eksperimen kedua memeriksa sistem bidang fase Fisher–KPP untuk \(c=1,2,3\), termasuk nilai eigen eksak, cabang menurun manifold tak stabil titik pelana, profil yang berosilasi dan menjadi negatif untuk \(c=1\), serta profil monoton tak negatif untuk kecepatan kritis \(c=2\) dan untuk \(c=3\).

**ID unit:** O005-LEGA-V101-CH09  
**ID notebook:** O005-LEGA-V101-CH09-NB01  
**Lisensi notebook:** CC BY-NC-SA 4.0  
**Provenans:** pendamping komputasi baru yang diturunkan dari persamaan Bab 9; tidak ada kode MATLAB, PPLANE, atau perangkat lunak berpemilik yang disalin.  
**Catatan perubahan:** notasi waktu tak berdimensi dipertahankan sebagai \(\tau\), sehingga koordinat gelombang berjalan ialah \(\xi=x-c\tau\); ambang muka populasi yang tak negatif ditulis \(c\geq2\), termasuk kasus kritis \(c=2\).


## Batas model dan pilihan yang dinyatakan

- Gerak acak berlangsung pada kisi persegi. Setiap langkah mempunyai panjang satu satuan dan memilih kanan, kiri, atas, atau bawah dengan peluang yang sama. Untuk \(L^2=X^2+Y^2\), teori memberi \(\mathbb{E}[L^2]=N\) setelah \(N\) langkah.
- Simulasi memakai 60.000 partikel, 400 langkah, dan biji generator bilangan acak yang dinyatakan di dalam sel. Toleransi statistik ditetapkan sebelum hasil dihitung dan diterapkan pada beberapa nilai \(N\).
- Persamaan Fisher–KPP tak berdimensi adalah \(n_\tau=n(1-n)+n_{xx}\). Dengan \(n(x,\tau)=v(\xi)\) dan \(\xi=x-c\tau\), sistem bidang fasenya ialah \(v'=w,\;w'=-cw-v(1-v)\).
- Integrasi numerik dimulai sangat dekat \(P_1=(1,0)\) pada arah nilai eigen positif titik pelana. Ini menghampiri cabang menurun manifold tak stabil pada jendela hingga; eksperimen numerik bukan pengganti bukti global tentang keberadaan atau pemilihan kecepatan muka.
- Semua parameter, keadaan awal, toleransi, kisi evaluasi, dan versi paket dinyatakan secara eksplisit. Tidak ada masukan interaktif, keadaan sesi tersembunyi, atau perbandingan piksel dengan gambar sumber.


In [ ]:
import platform
import time

import numpy as np
import scipy
from scipy.integrate import solve_ivp
import matplotlib
import matplotlib.pyplot as plt

np.set_printoptions(precision=10, suppress=True)

VERSI_REFERENSI = {
    "Python": "3.13.9",
    "NumPy": "2.4.4",
    "SciPy": "1.17.1",
    "Matplotlib": "3.10.9",
}
versi_aktual = {
    "Python": platform.python_version(),
    "NumPy": np.__version__,
    "SciPy": scipy.__version__,
    "Matplotlib": matplotlib.__version__,
}
assert versi_aktual == VERSI_REFERENSI, (versi_aktual, VERSI_REFERENSI)

def selesaikan(fun, rentang, awal, args=(), t_eval=None, max_step=np.inf):
    awal = np.asarray(awal, dtype=float)
    hasil = solve_ivp(
        fun,
        rentang,
        awal,
        args=args,
        method="DOP853",
        rtol=1e-9,
        atol=1e-11,
        t_eval=t_eval,
        max_step=max_step,
    )
    assert hasil.success, hasil.message
    assert hasil.y.shape[0] == awal.size
    assert hasil.t.size > 1 and np.all(np.diff(hasil.t) > 0.0)
    assert np.all(np.isfinite(hasil.t)) and np.all(np.isfinite(hasil.y))
    return hasil

def akhiri_gambar(fig):
    fig.canvas.draw()
    assert len(fig.axes) > 0
    if matplotlib.get_backend().lower() == "agg":
        plt.close(fig)
    else:
        plt.show()

assert isinstance(matplotlib.get_backend(), str) and matplotlib.get_backend()
assert np.finfo(float).eps < 3e-16
assert scipy.integrate.solve_ivp is solve_ivp

waktu_mulai_notebook = time.perf_counter()
print("Versi tervalidasi:", versi_aktual)
print("Backend Matplotlib:", matplotlib.get_backend())


## 1. Gerak acak kisi dan rata-rata kuadrat jarak

Tuliskan posisi partikel setelah \(N\) langkah sebagai \(\mathbf r_N=\sum_{k=1}^N\Delta\mathbf r_k\). Setiap inkremen mempunyai panjang satu, rata-rata nol, dan bebas dari inkremen lain. Karena itu,

\[
\mathbb E[L_N^2]
=\mathbb E[\mathbf r_N\cdot\mathbf r_N]
=\sum_{k=1}^N\mathbb E[|\Delta\mathbf r_k|^2]
+2\sum_{i<j}\mathbb E[\Delta\mathbf r_i\cdot\Delta\mathbf r_j]
=N.
\]

Simulasi berikut merekam seluruh kurva \(\mathbb E[L^2]\) terhadap \(N\), tetapi menyimpan hanya 1.000 posisi pertama pada langkah ke-200 untuk plot awan partikel. Label sumbu secara sengaja menyatakan rata-rata dari \(L^2\).


In [ ]:
BIJI_RNG = 19620260822
jumlah_partikel = 60_000
langkah_maksimum = 400
langkah_awan = 200

rng = np.random.default_rng(BIJI_RNG)
posisi_x = np.zeros(jumlah_partikel, dtype=np.int64)
posisi_y = np.zeros(jumlah_partikel, dtype=np.int64)
jumlah_arah = np.zeros(4, dtype=np.int64)
msd = np.empty(langkah_maksimum, dtype=float)
awan_200 = None

for N in range(1, langkah_maksimum + 1):
    arah = rng.integers(0, 4, size=jumlah_partikel, dtype=np.int8)
    jumlah_arah += np.bincount(arah, minlength=4)
    posisi_x += (arah == 0).astype(np.int64) - (arah == 1).astype(np.int64)
    posisi_y += (arah == 2).astype(np.int64) - (arah == 3).astype(np.int64)
    jarak_kuadrat = posisi_x * posisi_x + posisi_y * posisi_y
    msd[N - 1] = np.mean(jarak_kuadrat)
    if N == langkah_awan:
        awan_200 = np.column_stack((posisi_x[:1000].copy(), posisi_y[:1000].copy()))

kisi_N = np.arange(1, langkah_maksimum + 1, dtype=float)
frekuensi_arah = jumlah_arah / jumlah_arah.sum()
N_uji = np.array([25, 50, 100, 200, 400])
rasio_msd = msd[N_uji - 1] / N_uji
kemiringan_nol = float(np.dot(kisi_N, msd) / np.dot(kisi_N, kisi_N))
rata_posisi_akhir = np.array([np.mean(posisi_x), np.mean(posisi_y)])
galat_rata_ternormalisasi = float(np.linalg.norm(rata_posisi_akhir) / np.sqrt(langkah_maksimum))

assert BIJI_RNG == 19620260822
assert jumlah_partikel == 60_000 and langkah_maksimum == 400
assert awan_200 is not None and awan_200.shape == (1000, 2)
assert posisi_x.shape == (jumlah_partikel,) and posisi_y.shape == (jumlah_partikel,)
assert np.issubdtype(posisi_x.dtype, np.integer) and np.issubdtype(posisi_y.dtype, np.integer)
assert jumlah_arah.sum() == jumlah_partikel * langkah_maksimum
assert np.all(jumlah_arah > 0)
assert np.max(np.abs(frekuensi_arah - 0.25)) < 0.001
assert msd.shape == (langkah_maksimum,) and np.all(np.isfinite(msd))
assert np.all(msd > 0.0)
assert rasio_msd.shape == N_uji.shape
assert np.max(np.abs(rasio_msd - 1.0)) < 0.02
assert 0.985 < kemiringan_nol < 1.015
assert abs(msd[langkah_awan - 1] - langkah_awan) < 5.0
assert abs(msd[-1] - langkah_maksimum) < 8.0
assert galat_rata_ternormalisasi < 0.02
assert np.max(np.abs(awan_200)) < 100
assert np.array_equal(N_uji, np.unique(N_uji))

fig, sumbu = plt.subplots(1, 2, figsize=(11.2, 4.8), constrained_layout=True)
sumbu[0].scatter(awan_200[:, 0], awan_200[:, 1], s=8, color="#D55E00", alpha=0.55)
sumbu[0].set(
    xlabel="posisi x",
    ylabel="posisi y",
    title="1.000 partikel setelah 200 langkah",
    aspect="equal",
)
sumbu[1].plot(kisi_N, msd, color="#0072B2", label="estimasi ensemble")
sumbu[1].plot(kisi_N, kisi_N, "--", color="#222222", label=r"teori $\mathbb{E}[L^2]=N$")
sumbu[1].set(
    xlabel="jumlah langkah, N",
    ylabel=r"rata-rata kuadrat jarak, $\mathbb{E}[L^2]$",
    title="Penskalaan difusif",
)
for ax in sumbu:
    ax.grid(alpha=0.2)
sumbu[1].legend()
akhiri_gambar(fig)

print("Frekuensi empat arah:", frekuensi_arah)
print("N yang diuji:", N_uji)
print("E[L^2]/N:", rasio_msd)
print(f"Kemiringan melalui titik asal = {kemiringan_nol:.9f}")
print("Rata-rata posisi akhir:", rata_posisi_akhir)


## 2. Fisher–KPP: titik tetap dan nilai eigen eksak

Untuk

\[
v'=w,\qquad w'=-cw-v(1-v),
\]

titik tetapnya adalah \(P_0=(0,0)\) dan \(P_1=(1,0)\), dengan

\[
J(v,w)=
\begin{pmatrix}
0&1\\
-1+2v&-c
\end{pmatrix}.
\]

Nilai eigen di \(P_0\) ialah

\[
\lambda_{0,\pm}=\frac{-c\pm\sqrt{c^2-4}}{2},
\]

sedangkan nilai eigen di titik pelana \(P_1\) ialah

\[
\lambda_{1,\pm}=\frac{-c\pm\sqrt{c^2+4}}{2}.
\]

Jadi \(c=1\) memberi spiral stabil di \(P_0\), \(c=2\) memberi nilai eigen ganda \(-1\) dengan hanya satu arah eigen, dan \(c=3\) memberi dua nilai eigen riil negatif. Titik \(P_1\) tetap berupa pelana dalam ketiga kasus.


In [ ]:
def fisher_kpp(xi, z, c):
    v, w = np.asarray(z, dtype=float)
    return np.array([w, -c * w - v * (1.0 - v)])

def jacobian_fisher(v, w, c):
    return np.array([[0.0, 1.0], [-1.0 + 2.0 * v, -c]])

def eigen_eksak_P0(c):
    akar = np.sqrt(complex(c * c - 4.0))
    return np.array([(-c + akar) / 2.0, (-c - akar) / 2.0])

def eigen_eksak_P1(c):
    akar = np.sqrt(c * c + 4.0)
    return np.array([(-c + akar) / 2.0, (-c - akar) / 2.0])

P0 = np.array([0.0, 0.0])
P1 = np.array([1.0, 0.0])
daftar_c = (1.0, 2.0, 3.0)
ringkasan_eigen = {}

for c in daftar_c:
    J0 = jacobian_fisher(*P0, c)
    J1 = jacobian_fisher(*P1, c)
    eig0 = np.linalg.eigvals(J0)
    eig1 = np.linalg.eigvals(J1)
    eksak0 = eigen_eksak_P0(c)
    eksak1 = eigen_eksak_P1(c)

    assert np.array_equal(fisher_kpp(0.0, P0, c), np.zeros(2))
    assert np.array_equal(fisher_kpp(0.0, P1, c), np.zeros(2))
    assert np.isclose(np.trace(J0), -c) and np.isclose(np.linalg.det(J0), 1.0)
    assert np.isclose(np.trace(J1), -c) and np.isclose(np.linalg.det(J1), -1.0)
    assert np.allclose(np.sort_complex(eig0), np.sort_complex(eksak0), atol=2e-12)
    assert np.allclose(np.sort_complex(eig1), np.sort_complex(eksak1), atol=2e-12)
    assert np.min(eig1.real) < 0.0 < np.max(eig1.real)
    assert np.all(eig0.real < 0.0)
    ringkasan_eigen[c] = (eig0, eig1)

eig0_c1 = ringkasan_eigen[1.0][0]
eig0_c2 = ringkasan_eigen[2.0][0]
eig0_c3 = ringkasan_eigen[3.0][0]
assert np.allclose(eig0_c1.real, -0.5)
assert np.allclose(np.sort(np.abs(eig0_c1.imag)), [np.sqrt(3.0) / 2.0] * 2)
assert np.allclose(eig0_c2, [-1.0, -1.0], atol=2e-12)
assert np.linalg.matrix_rank(jacobian_fisher(*P0, 2.0) + np.eye(2), tol=1e-12) == 1
assert np.allclose(
    np.sort(eig0_c3),
    np.sort([(-3.0 - np.sqrt(5.0)) / 2.0, (-3.0 + np.sqrt(5.0)) / 2.0]),
)
assert np.all(np.isreal(eig0_c3)) and np.all(eig0_c3 < 0.0)

for c in daftar_c:
    lambda_tak_stabil = float(np.max(eigen_eksak_P1(c)))
    arah_tak_stabil = np.array([1.0, lambda_tak_stabil])
    assert lambda_tak_stabil > 0.0
    assert np.linalg.norm(
        jacobian_fisher(*P1, c) @ arah_tak_stabil - lambda_tak_stabil * arah_tak_stabil
    ) < 2e-14

print("Nilai eigen P0 dan P1:")
for c in daftar_c:
    print(f"  c={c:g}: P0={ringkasan_eigen[c][0]}, P1={ringkasan_eigen[c][1]}")


## 3. Cabang manifold tak stabil dan profil gelombang

Di \(P_1\), nilai eigen positif adalah \(\lambda_u=(-c+\sqrt{c^2+4})/2\), dengan vektor eigen \((1,\lambda_u)^T\). Keadaan awal

\[
(v,w)=(1,0)-\varepsilon(1,\lambda_u),\qquad \varepsilon=10^{-6},
\]

memilih cabang yang bergerak menuju nilai \(v\) lebih kecil. Integrasi dilakukan ke arah \(\xi\) yang meningkat. Agar profil dapat dibandingkan, masing-masing sumbu \(\xi\) digeser sehingga perpotongan pertama \(v=1/2\) berada di nol.

Untuk \(c=1\), pendekatan menuju \(P_0\) berpilin sehingga \(v\) berulang kali melintasi nol. Profil itu merupakan solusi matematis sistem bidang fase, tetapi tidak dapat menjadi kepadatan populasi. Untuk \(c=2\), nilai eigen ganda \(-1\) menghasilkan muka kritis yang tetap monoton dan tak negatif. Untuk \(c=3\), ekor muka mengikuti nilai eigen lambat \((-3+\sqrt5)/2\).


In [ ]:
epsilon_manifold = 1e-6
kisi_xi = np.linspace(0.0, 100.0, 5001)
hasil_fisher = {}
diagnostik_fisher = {}

for c in daftar_c:
    lambda_u = float(np.max(eigen_eksak_P1(c)))
    arah_u = np.array([1.0, lambda_u])
    awal = P1 - epsilon_manifold * arah_u
    prediksi_linear = -epsilon_manifold * lambda_u * arah_u
    residu_linear = np.linalg.norm(fisher_kpp(0.0, awal, c) - prediksi_linear)

    assert np.isclose(awal[0], 1.0 - epsilon_manifold)
    assert awal[1] < 0.0
    assert residu_linear < 3.0 * epsilon_manifold**2

    hasil = selesaikan(
        fisher_kpp,
        (0.0, 100.0),
        awal,
        args=(c,),
        t_eval=kisi_xi,
        max_step=0.08,
    )
    v, w = hasil.y
    indeks_tengah = int(np.flatnonzero(v <= 0.5)[0])
    xi_kiri, xi_kanan = hasil.t[indeks_tengah - 1:indeks_tengah + 1]
    v_kiri, v_kanan = v[indeks_tengah - 1:indeks_tengah + 1]
    xi_tengah = float(xi_kiri + (0.5 - v_kiri) * (xi_kanan - xi_kiri) / (v_kanan - v_kiri))
    lintas_nol = np.flatnonzero(v[:-1] * v[1:] < 0.0)
    ekstrem = np.flatnonzero(w[:-1] * w[1:] < 0.0)

    assert hasil.t[0] == 0.0 and hasil.t[-1] == 100.0
    assert 0 < indeks_tengah < hasil.t.size
    assert 0.0 < xi_tengah < 100.0
    assert np.linalg.norm(hasil.y[:, -1]) < 2e-8
    assert np.min(v) > -0.1 and np.max(v) <= 1.0
    assert np.min(w) > -0.3 and np.max(w) < 0.05

    hasil_fisher[c] = hasil
    diagnostik_fisher[c] = {
        "xi_setengah": xi_tengah,
        "minimum_v": float(np.min(v)),
        "lintas_nol": int(lintas_nol.size),
        "ekstrem": int(ekstrem.size),
        "norma_akhir": float(np.linalg.norm(hasil.y[:, -1])),
        "residu_awal": float(residu_linear),
    }

v1, w1 = hasil_fisher[1.0].y
lintas_nol_c1 = np.flatnonzero(v1[:-1] * v1[1:] < 0.0)
ekstrem_c1 = np.flatnonzero(w1[:-1] * w1[1:] < 0.0)
assert np.min(v1) < -0.04
assert np.max(w1) > 0.02
assert lintas_nol_c1.size >= 8
assert ekstrem_c1.size >= 8
assert hasil_fisher[1.0].t[lintas_nol_c1[0]] > diagnostik_fisher[1.0]["xi_setengah"]
assert diagnostik_fisher[1.0]["norma_akhir"] < 1e-10

v2, w2 = hasil_fisher[2.0].y
mask_ekor_c2 = (v2 > 1e-8) & (v2 < 1e-4)
laju_ekor_c2 = float(np.median(w2[mask_ekor_c2] / v2[mask_ekor_c2]))
assert np.min(v2) >= -1e-12
assert np.max(w2) <= 1e-12
assert np.max(np.diff(v2)) <= 1e-12
assert v2[-1] < 1e-12
assert np.count_nonzero(mask_ekor_c2) > 100
assert -1.0 < laju_ekor_c2 < -0.88

v3, w3 = hasil_fisher[3.0].y
mask_ekor_c3 = (v3 > 1e-8) & (v3 < 1e-4)
laju_ekor_c3 = float(np.median(w3[mask_ekor_c3] / v3[mask_ekor_c3]))
lambda_lambat_c3 = (-3.0 + np.sqrt(5.0)) / 2.0
assert np.min(v3) >= -1e-12
assert np.max(w3) <= 1e-12
assert np.max(np.diff(v3)) <= 1e-12
assert v3[-1] < 2e-8
assert np.count_nonzero(mask_ekor_c3) > 100
assert abs(laju_ekor_c3 - lambda_lambat_c3) < 2e-4

v_grid = np.linspace(-0.2, 1.2, 29)
w_grid = np.linspace(-0.4, 0.4, 25)
V, W = np.meshgrid(v_grid, w_grid)
assert V.shape == W.shape == (25, 29)

fig_fase, sumbu_fase = plt.subplots(1, 3, figsize=(14.0, 4.6), constrained_layout=True)
for ax, c in zip(sumbu_fase, daftar_c):
    dV = W
    dW = -c * W - V * (1.0 - V)
    norma = np.hypot(dV, dW)
    dV_unit = np.divide(dV, norma, out=np.zeros_like(dV), where=norma > 0.0)
    dW_unit = np.divide(dW, norma, out=np.zeros_like(dW), where=norma > 0.0)
    assert np.all(np.isfinite(dV_unit)) and np.all(np.isfinite(dW_unit))
    ax.quiver(V, W, dV_unit, dW_unit, color="#777777", alpha=0.65, pivot="mid")
    hasil = hasil_fisher[c]
    ax.plot(hasil.y[0], hasil.y[1], color="#0072B2", linewidth=2.4, label="cabang dari P1")
    ax.plot(*P0, "o", color="#D55E00", label="P0")
    ax.plot(*P1, "s", color="#009E73", label="P1")
    ax.set(
        xlim=(-0.2, 1.2),
        ylim=(-0.4, 0.4),
        xlabel="v",
        ylabel="w",
        title=f"Potret fase, c={c:g}",
    )
    ax.grid(alpha=0.18)
    ax.legend(fontsize=8)
akhiri_gambar(fig_fase)

fig_profil, ax_profil = plt.subplots(figsize=(9.2, 5.2), constrained_layout=True)
for c, warna in zip(daftar_c, ("#D55E00", "#0072B2", "#009E73")):
    hasil = hasil_fisher[c]
    xi_geser = hasil.t - diagnostik_fisher[c]["xi_setengah"]
    ax_profil.plot(xi_geser, hasil.y[0], color=warna, label=f"c={c:g}")
ax_profil.axhline(0.0, color="#222222", linewidth=0.8)
ax_profil.axhline(1.0, color="#222222", linewidth=0.8, linestyle="--")
ax_profil.set(
    xlim=(-18.0, 38.0),
    xlabel=r"koordinat bergerak, $\xi-\xi_{1/2}$",
    ylabel="profil v",
    title="Profil muka Fisher–KPP dari cabang manifold tak stabil",
)
ax_profil.grid(alpha=0.2)
ax_profil.legend()
akhiri_gambar(fig_profil)

waktu_notebook = time.perf_counter() - waktu_mulai_notebook
assert waktu_notebook < 120.0

print("Diagnostik profil Fisher–KPP:")
for c in daftar_c:
    print(f"  c={c:g}: {diagnostik_fisher[c]}")
print(f"Laju ekor c=2 = {laju_ekor_c2:.9f}")
print(f"Laju ekor c=3 = {laju_ekor_c3:.9f}; nilai eigen lambat = {lambda_lambat_c3:.9f}")
print(f"Waktu eksekusi sel komputasi = {waktu_notebook:.3f} s")


## Kesimpulan

Eksperimen gerak acak menguji hukum \(\mathbb E[L^2]=N\) pada lima jumlah langkah dan pada keseluruhan kurva melalui kemiringan kuadrat terkecil yang dipaksa melewati titik asal. Grafik menggunakan label “rata-rata kuadrat jarak” agar tidak tertukar dengan \((\mathbb E[L])^2\).

Analisis Fisher–KPP memeriksa nilai eigen eksak di kedua titik tetap sebelum melakukan integrasi. Cabang menurun dari \(P_1\) mendekati spiral \(P_0\) sambil melintasi \(v=0\) berulang kali untuk \(c=1\), sehingga profilnya tidak sah sebagai kepadatan. Pada \(c=2\), \(P_0\) mempunyai nilai eigen ganda \(-1\) dan profil kritis tetap monoton tak negatif. Pada \(c=3\), profil juga monoton tak negatif dan laju ekor numeriknya cocok dengan nilai eigen lambat \((-3+\sqrt5)/2\).

Hasil ini membuat permukaan komputasi Bab 9 dapat direproduksi secara terbuka tanpa bergantung pada GUI MATLAB atau PPLANE. Simulasi tidak membuktikan teorema keberadaan global dan tidak menyatakan bahwa persamaan diferensial parsial selalu memilih satu kecepatan tertentu; pemilihan kecepatan juga bergantung pada data awal.
